In [ ]:
!git clone https://github.com/kaizar-rang/yanglab-acne.git
%cd yanglab-acne
%pip install roboflow ultralytics -q
%pip install -r requirements.txt -q
print("\nDone. Restart the runtime (Runtime -> Restart session), then continue from Cell 2.")

In [ ]:
import os
import json
import getpass
from pathlib import Path

%cd /content/yanglab-acne

roboflow_key = getpass.getpass("Enter ROBOFLOW_API_KEY: ")
os.environ["ROBOFLOW_API_KEY"] = roboflow_key

from roboflow import Roboflow
rf = Roboflow(api_key=roboflow_key)
project = rf.workspace("acne-vulgaris-detection").project("acne04-detection")
dataset = project.version(1).download("yolov8", location="data/acne04_yolov8")

print("Download complete.")

In [ ]:
import torch
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Verify dataset
data_yaml = Path("data/acne04_yolov8/data.yaml")
print(f"\ndata.yaml found: {data_yaml.exists()}")

In [ ]:
from ultralytics import YOLO

# Load YOLOv8s pretrained on COCO
model = YOLO("yolov8s.pt")

results = model.train(
    data="data/acne04_yolov8/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    lr0=0.01,
    momentum=0.9,
    weight_decay=0.0005,
    project="outputs/yolov8",
    name="acne04",
    exist_ok=True,
    verbose=True
)

print("\nTraining complete.")

In [ ]:
from ultralytics import YOLO

model = YOLO("outputs/yolov8/acne04/weights/best.pt")

metrics = model.val(
    data="data/acne04_yolov8/data.yaml",
    imgsz=640,
    split="val"
)

print(f"mAP@0.5:      {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.3f}")
print(f"Precision:    {metrics.box.mp:.3f}")
print(f"Recall:       {metrics.box.mr:.3f}")

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

model = YOLO("outputs/yolov8/acne04/weights/best.pt")

val_images = list(Path("data/acne04_yolov8/valid/images").glob("*.jpg"))
sample = random.sample(val_images, 6)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample):
    result = model(str(img_path), verbose=False)[0]
    annotated = result.plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    ax.axis("off")
    ax.set_title(img_path.name, fontsize=8)

plt.suptitle("YOLOv8s Predictions on ACNE04 Validation Set", fontsize=13)
plt.tight_layout()
plt.savefig("outputs/yolov8/acne04/sample_predictions.png", dpi=150)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

model = YOLO("outputs/yolov8/acne04/weights/best.pt")

metrics = model.val(
    data="data/acne04_yolov8/data.yaml",
    imgsz=640,
    split="val",
    plots=True  # automatically saves confusion matrix and PR curve
)

# Load and display the saved confusion matrix
conf_matrix_path = "outputs/yolov8/acne04/confusion_matrix_normalized.png"
img = plt.imread(conf_matrix_path)
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis("off")
plt.title("YOLOv8 Normalized Confusion Matrix")
plt.tight_layout()
plt.savefig("outputs/yolov8/acne04/confusion_matrix_display.png", dpi=150)
plt.show()

In [ ]:
# Load and display the saved PR curve
pr_curve_path = "outputs/yolov8/acne04/PR_curve.png"
img = plt.imread(pr_curve_path)
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis("off")
plt.title("YOLOv8 Precision-Recall Curve")
plt.tight_layout()
plt.savefig("outputs/yolov8/acne04/pr_curve_display.png", dpi=150)
plt.show()

In [ ]:
from google.colab import files
import os

os.makedirs("outputs/yolov8/acne04", exist_ok=True)

!zip -r yolov8_results.zip \
    outputs/yolov8/acne04/weights/best.pt \
    outputs/yolov8/acne04/sample_predictions.png \
    outputs/yolov8/acne04/results.csv

files.download("yolov8_results.zip")
print("Download initiated.")